# Lesson 9: Grids and diffusion

**How two chemicals grow spots and stripes** · Advanced · from [Hypnotic Patterns](https://hypnoticpatterns.doner.cloud)

Two chemicals share a dish. One is food, the other eats it to make more of itself, and both spread out. From a single drop, the second grows into spots, stripes or a maze all by itself, like the markings on an animal’s skin.

Watch the films and play with the live demo: [Turing patterns](https://hypnoticpatterns.doner.cloud/pattern/turing-patterns). The same lesson with a quiz: [https://hypnoticpatterns.doner.cloud/learn/turing-patterns](https://hypnoticpatterns.doner.cloud/learn/turing-patterns).

## What you will learn

- 2-D arrays as a grid of cells
- np.roll for neighbours, with edges that wrap around
- the Laplacian: a cell against its neighbours
- updating a whole grid in one line
- showing an array with plt.imshow

## The script

Run the cell below (Shift + Enter). numpy and matplotlib are already in Colab.

In [ ]:
"""Turing patterns: the Gray-Scott reaction-diffusion rule on a 128 x 128 grid.
Run: python turing-patterns.py   (needs: pip install numpy matplotlib; takes a few seconds)
"""
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")

size, steps = 128, 6000
f, k = 0.0545, 0.062  # feed and kill rates: coral
DA, DB = 1.0, 0.5  # A spreads twice as fast as B

A, B = np.ones((size, size)), np.zeros((size, size))
rng = np.random.default_rng()
for y, x in rng.integers(10, size - 18, (6, 2)):
    B[y : y + 8, x : x + 8] = 1.0  # six drops of B


def laplacian(Z):
    """Each cell against its neighbours: 0.2 for the four sides, 0.05 for the corners (Karl Sims)"""
    sides = np.roll(Z, 1, 0) + np.roll(Z, -1, 0) + np.roll(Z, 1, 1) + np.roll(Z, -1, 1)
    corners = (np.roll(np.roll(Z, 1, 0), 1, 1) + np.roll(np.roll(Z, 1, 0), -1, 1)
               + np.roll(np.roll(Z, -1, 0), 1, 1) + np.roll(np.roll(Z, -1, 0), -1, 1))
    return 0.2 * sides + 0.05 * corners - Z


for _ in range(steps):
    reaction = A * B * B  # one A and two B make three B
    A += DA * laplacian(A) - reaction + f * (1 - A)
    B += DB * laplacian(B) + reaction - (f + k) * B

plt.figure(figsize=(6, 6))
plt.imshow(B, cmap="magma", vmin=0, vmax=0.5)
plt.axis("off")
plt.show()

# Try: f, k = 0.039, 0.058 (a cell net) or 0.034, 0.063 (spots) -> the same rule, another pattern

## Line by line

1. `A, B = np.ones((size, size)), np.zeros((size, size))`  
   Two 128 × 128 grids: the food A everywhere, no B yet.
2. `B[y : y + 8, x : x + 8] = 1.0`  
   A drop of B: an 8 × 8 square of the grid, set at once by slicing.
3. `def laplacian(Z):`  
   How much a cell differs from its neighbours: positive where the neighbours have more, so diffusion fills it up.
4. `sides = np.roll(Z, 1, 0) + np.roll(Z, -1, 0) + np.roll(Z, 1, 1) + np.roll(Z, -1, 1)`  
   np.roll shifts the whole grid by one cell up, down, left or right; the edges wrap around, so the dish has no border.
5. `reaction = A * B * B`  
   The reaction, one A and two B make three B, for every cell at once.
6. `A += DA * laplacian(A) - reaction + f * (1 - A)`  
   The Gray–Scott rule for A: it spreads, is eaten, and is fed back towards 1 at the rate f.
7. `plt.imshow(B, cmap="magma", vmin=0, vmax=0.5)`  
   The grid of B as a picture, one pixel per cell.

## Try this

Try f, k = 0.039, 0.058 for a cell net or 0.034, 0.063 for spots: the same rule, another species.

## Exercises

Write your code in the empty cells. The hints and answers are at the end of the notebook.

1. Try f, k = 0.034, 0.063.
2. Make B spread as fast as A: DB = 1.0. What happens?
3. Use one drop in the middle instead of six.

**Exercise 1.** Try f, k = 0.034, 0.063.

In [ ]:
# your code here

**Exercise 2.** Make B spread as fast as A: DB = 1.0. What happens?

In [ ]:
# your code here

**Exercise 3.** Use one drop in the middle instead of six.

In [ ]:
# your code here

## Why it works

Chemical A is fed in everywhere at a slow rate f. Chemical B turns A into more B (one A and two B make three B) and is removed at a rate k. Both spread, but A spreads twice as fast as B.

That difference is the trick. B grows where it already is and uses up the food. Because A moves fast, the shortage spreads too, into a ring around the patch where new B cannot start. Patches with gaps between them: a pattern with its own spacing.

These are the Gray–Scott equations below. Alan Turing showed in 1952 that diffusion, which usually smooths things out, can make an even mixture unstable when one substance spreads faster than the other. John Pearson (1993) mapped how small changes of f and k give spots that divide, stripes, mazes or turbulence; the five presets of the demo are among them.

Each pixel of the demo is one cell of a grid, updated many times a second by the same rule, with the values Karl Sims uses: D_A = 1 and D_B = 0.5.

`∂A/∂t = D_A·∇²A − A·B² + f·(1 − A)      ∂B/∂t = D_B·∇²B + A·B² − (f + k)·B`

## Hints and answers

1. *Hint:* One line near the top.  
   *Answer:* Spots: round patches of B, evenly spaced, instead of the coral’s winding lines.
2. *Hint:* DA, DB = 1.0, 1.0.  
   *Answer:* No pattern at all: the drops fade away to nothing. Without A spreading faster than B, nothing holds the patches apart. This is Turing’s point.
3. *Hint:* Replace the loop with B[60:68, 60:68] = 1.0.  
   *Answer:* The coral grows outwards from the one drop with a fourfold symmetry that copies the square drop: one seed gives an ordered pattern, six random drops a natural-looking one.

## Sources

- [Turing pattern (Wikipedia)](https://en.wikipedia.org/wiki/Turing_pattern)
- [A. M. Turing, “The chemical basis of morphogenesis”, Phil. Trans. R. Soc. B 237 (1952)](https://doi.org/10.1098/rstb.1952.0012)
- [J. E. Pearson, “Complex patterns in a simple system”, Science 261 (1993)](https://doi.org/10.1126/science.261.5118.189)
- [Karl Sims: Reaction-Diffusion Tutorial](https://www.karlsims.com/rd.html)

Free to use for learning and in class. Made for the YouTube channel [Hypnotic Patterns](https://www.youtube.com/@HypnoticPatternsAudio).